In [5]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile

data_path = r"C:\Users\egkbo\dallas_heat_island\data\raw"
envi_path = os.path.join(data_path, "enviroatlas", "CONUS_metrics_CSV", "CONUS_metrics_CSV")

# reload all data
income = pd.read_csv(os.path.join(data_path, "ACSDT5Y2023.B19013-Data.csv"), skiprows=1)
population = pd.read_csv(os.path.join(data_path, "ACSDT5Y2023.B01003-Data.csv"), skiprows=1)
housing = pd.read_csv(os.path.join(data_path, "ACSDT5Y2023.B25034-Data.csv"), skiprows=1)
dallas_tracts = gpd.read_file(os.path.join(data_path, "tl_2022_48_tract.zip"))
dallas_tracts = dallas_tracts[dallas_tracts['COUNTYFP'] == '113']
impervious = pd.read_csv(os.path.join(envi_path, "Impervious_CONUS.csv"))
canopy = pd.read_csv(os.path.join(envi_path, "PCanopy_CONUS.csv"))

with rasterio.open(os.path.join(data_path, "LC09_L2SP_027035_20230819_20230821_02_T1_ST_B10.TIF")) as src:
    lst_data = src.read(1).astype("float32")
    lst_meta = src.meta
    lst_transform = src.transform
    lst_crs = src.crs

with rasterio.open(os.path.join(data_path, "LC09_L2SP_027035_20230819_20230821_02_T1_SR_B4.TIF")) as src:
    b4_data = src.read(1).astype("float32")

with rasterio.open(os.path.join(data_path, "LC09_L2SP_027035_20230819_20230821_02_T1_SR_B5.TIF")) as src:
    b5_data = src.read(1).astype("float32")

print("All data reloaded successfully!")

All data reloaded successfully!


In [6]:
#look at income
print("INCOME")
print(income.info())
print("\n")
print(income.describe())
print("\n")
print(income.head())

INCOME
<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 5 columns):
 #   Column                                                                                               Non-Null Count  Dtype  
---  ------                                                                                               --------------  -----  
 0   Geography                                                                                            645 non-null    str    
 1   Geographic Area Name                                                                                 645 non-null    str    
 2   Estimate!!Median household income in the past 12 months (in 2023 inflation-adjusted dollars)         645 non-null    str    
 3   Margin of Error!!Median household income in the past 12 months (in 2023 inflation-adjusted dollars)  645 non-null    str    
 4   Unnamed: 4                                                                                           0 non-null    

In [7]:
#look at population
print("POPULATION")
print(population.info())
print("\n")
print(population.head())

POPULATION
<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Geography               645 non-null    str    
 1   Geographic Area Name    645 non-null    str    
 2   Estimate!!Total         645 non-null    int64  
 3   Margin of Error!!Total  645 non-null    int64  
 4   Unnamed: 4              0 non-null      float64
dtypes: float64(1), int64(2), str(2)
memory usage: 25.3 KB
None


              Geography                     Geographic Area Name  \
0  1400000US48113000100     Census Tract 1; Dallas County; Texas   
1  1400000US48113000201  Census Tract 2.01; Dallas County; Texas   
2  1400000US48113000202  Census Tract 2.02; Dallas County; Texas   
3  1400000US48113000300     Census Tract 3; Dallas County; Texas   
4  1400000US48113000401  Census Tract 4.01; Dallas County; Texas   

   Estimate!!Total  Margin of Error!!Total  Unname

In [8]:
print("HOUSING")
print(housing.info())
print("\n")
print(housing.head())

HOUSING
<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 25 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Geography                                       645 non-null    str    
 1   Geographic Area Name                            645 non-null    str    
 2   Estimate!!Total:                                645 non-null    int64  
 3   Margin of Error!!Total:                         645 non-null    int64  
 4   Estimate!!Total:!!Built 2020 or later           645 non-null    int64  
 5   Margin of Error!!Total:!!Built 2020 or later    645 non-null    int64  
 6   Estimate!!Total:!!Built 2010 to 2019            645 non-null    int64  
 7   Margin of Error!!Total:!!Built 2010 to 2019     645 non-null    int64  
 8   Estimate!!Total:!!Built 2000 to 2009            645 non-null    int64  
 9   Margin of Error!!Total:!!Built 2000 to 2009   

In [9]:
print("DALLAS SHAPEFILE")
print(dallas_tracts.info())
print("\n")
print(dallas_tracts.head())

DALLAS SHAPEFILE
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 645 entries, 176 to 6796
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   STATEFP   645 non-null    str     
 1   COUNTYFP  645 non-null    str     
 2   TRACTCE   645 non-null    str     
 3   GEOID     645 non-null    str     
 4   NAME      645 non-null    str     
 5   NAMELSAD  645 non-null    str     
 6   MTFCC     645 non-null    str     
 7   FUNCSTAT  645 non-null    str     
 8   ALAND     645 non-null    int64   
 9   AWATER    645 non-null    int64   
 10  INTPTLAT  645 non-null    str     
 11  INTPTLON  645 non-null    str     
 12  geometry  645 non-null    geometry
dtypes: geometry(1), int64(2), str(10)
memory usage: 70.5 KB
None


    STATEFP COUNTYFP TRACTCE        GEOID    NAME             NAMELSAD  MTFCC  \
176      48      113  016638  48113016638  166.38  Census Tract 166.38  G5020   
264      48      113  013630  48113013630 

In [10]:
print("IMPERVIOUS")
print(impervious.describe())
print("\n")
print("=== CANOPY ===")
print(canopy.describe())

IMPERVIOUS
           OBJECTID        HUC_12         PIMPV
count  82915.000000  8.291500e+04  82871.000000
mean   41458.000000  1.002108e+11      1.963675
std    23935.643122  4.997070e+10      5.384863
min        1.000000  1.010002e+10      0.000000
25%    20729.500000  5.120204e+10      0.180947
50%    41458.000000  1.018001e+11      0.541063
75%    62186.500000  1.407000e+11      1.342630
max    82915.000000  1.810020e+11     82.318146


=== CANOPY ===
           OBJECTID        HUC_12       pCanopy
count  82915.000000  8.291500e+04  82915.000000
mean   41458.000000  1.002108e+11     24.385275
std    23935.643122  4.997070e+10     26.255168
min        1.000000  1.010002e+10      0.000000
25%    20729.500000  5.120204e+10      0.642125
50%    41458.000000  1.018001e+11     13.164774
75%    62186.500000  1.407000e+11     45.953592
max    82915.000000  1.810020e+11     95.899207


In [11]:
print("LANDSAT LST (raw)")
print(f"Shape: {lst_data.shape}")
print(f"Min value: {lst_data.min()}")
print(f"Max value: {lst_data.max()}")
print(f"Mean value: {lst_data.mean():.2f}")
print(f"Zero pixels (no data): {(lst_data == 0).sum()}")

print("\n=== NDVI (preview) ===")
ndvi_preview = (b5_data - b4_data) / (b5_data + b4_data)
print(f"NDVI min: {ndvi_preview.min():.3f}")
print(f"NDVI max: {ndvi_preview.max():.3f}")
print(f"NDVI mean: {ndvi_preview.mean():.3f}")

LANDSAT LST (raw)
Shape: (7691, 7571)


Min value: 0.0
Max value: 54823.0
Mean value: 33184.50
Zero pixels (no data): 17472931

=== NDVI (preview) ===
NDVI min: nan
NDVI max: nan
NDVI mean: nan


C:\Users\egkbo\AppData\Local\Temp\ipykernel_55144\3172492674.py:9: RuntimeWarning: invalid value encountered in divide
  ndvi_preview = (b5_data - b4_data) / (b5_data + b4_data)


In [12]:
# clean income
income_clean = income.copy()

# drop unneeded columns
income_clean = income_clean.drop(columns=["Unnamed: 4", 
    "Margin of Error!!Median household income in the past 12 months (in 2023 inflation-adjusted dollars)"])

# rename columns
income_clean.columns = ["geoid", "tract_name", "median_income"]

# extract just the FIPS code from geoid (remove "1400000US" prefix)
income_clean["geoid"] = income_clean["geoid"].str.replace("1400000US", "")

# convert income to numeric (some values may be "-" or "N/A")
income_clean["median_income"] = pd.to_numeric(income_clean["median_income"], errors="coerce")

print("Income cleaned:")
print(income_clean.info())
print("\n")
print(income_clean.head())
print("\nMissing values:", income_clean["median_income"].isna().sum())

Income cleaned:
<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   geoid          645 non-null    str    
 1   tract_name     645 non-null    str    
 2   median_income  630 non-null    float64
dtypes: float64(1), str(2)
memory usage: 15.2 KB
None


         geoid                               tract_name  median_income
0  48113000100     Census Tract 1; Dallas County; Texas       178472.0
1  48113000201  Census Tract 2.01; Dallas County; Texas       195000.0
2  48113000202  Census Tract 2.02; Dallas County; Texas       147708.0
3  48113000300     Census Tract 3; Dallas County; Texas       100409.0
4  48113000401  Census Tract 4.01; Dallas County; Texas        58232.0

Missing values: 15


In [13]:
# clean population
population_clean = population.copy()

# drop unneeded columns
population_clean = population_clean.drop(columns=["Unnamed: 4",
    "Margin of Error!!Total",
    "Geographic Area Name"])

# rename columns
population_clean.columns = ["geoid", "total_population"]

# extract FIPS code
population_clean["geoid"] = population_clean["geoid"].str.replace("1400000US", "")

print("Population cleaned:")
print(population_clean.info())
print("\n")
print(population_clean.head())
print("\nMissing values:", population_clean["total_population"].isna().sum())

Population cleaned:
<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   geoid             645 non-null    str  
 1   total_population  645 non-null    int64
dtypes: int64(1), str(1)
memory usage: 10.2 KB
None


         geoid  total_population
0  48113000100              4450
1  48113000201              3088
2  48113000202              3829
3  48113000300              4133
4  48113000401              4666

Missing values: 0


In [14]:
print(housing.columns.tolist())


['Geography', 'Geographic Area Name', 'Estimate!!Total:', 'Margin of Error!!Total:', 'Estimate!!Total:!!Built 2020 or later', 'Margin of Error!!Total:!!Built 2020 or later', 'Estimate!!Total:!!Built 2010 to 2019', 'Margin of Error!!Total:!!Built 2010 to 2019', 'Estimate!!Total:!!Built 2000 to 2009', 'Margin of Error!!Total:!!Built 2000 to 2009', 'Estimate!!Total:!!Built 1990 to 1999', 'Margin of Error!!Total:!!Built 1990 to 1999', 'Estimate!!Total:!!Built 1980 to 1989', 'Margin of Error!!Total:!!Built 1980 to 1989', 'Estimate!!Total:!!Built 1970 to 1979', 'Margin of Error!!Total:!!Built 1970 to 1979', 'Estimate!!Total:!!Built 1960 to 1969', 'Margin of Error!!Total:!!Built 1960 to 1969', 'Estimate!!Total:!!Built 1950 to 1959', 'Margin of Error!!Total:!!Built 1950 to 1959', 'Estimate!!Total:!!Built 1940 to 1949', 'Margin of Error!!Total:!!Built 1940 to 1949', 'Estimate!!Total:!!Built 1939 or earlier', 'Margin of Error!!Total:!!Built 1939 or earlier', 'Unnamed: 24']


In [15]:
# clean housing
housing_clean = housing.copy()

# drop margin of error columns and unnamed
cols_to_drop = ["Unnamed: 24", "Geographic Area Name",
    "Margin of Error!!Total:",
    "Margin of Error!!Total:!!Built 2020 or later",
    "Margin of Error!!Total:!!Built 2010 to 2019",
    "Margin of Error!!Total:!!Built 2000 to 2009",
    "Margin of Error!!Total:!!Built 1990 to 1999",
    "Margin of Error!!Total:!!Built 1980 to 1989",
    "Margin of Error!!Total:!!Built 1970 to 1979",
    "Margin of Error!!Total:!!Built 1960 to 1969",
    "Margin of Error!!Total:!!Built 1950 to 1959",
    "Margin of Error!!Total:!!Built 1940 to 1949",
    "Margin of Error!!Total:!!Built 1939 or earlier"]

housing_clean = housing_clean.drop(columns=cols_to_drop)

# rename columns
housing_clean.columns = ["geoid", "total_units", "built_2020plus",
    "built_2010_2019", "built_2000_2009", "built_1990_1999",
    "built_1980_1989", "built_1970_1979", "built_1960_1969",
    "built_1950_1959", "built_1940_1949", "built_1939_earlier"]

# extract FIPS code
housing_clean["geoid"] = housing_clean["geoid"].str.replace("1400000US", "")

# create pct_old_housing (built before 1980)
old_cols = ["built_1970_1979", "built_1960_1969", "built_1950_1959",
            "built_1940_1949", "built_1939_earlier"]
housing_clean["old_housing_units"] = housing_clean[old_cols].sum(axis=1)
housing_clean["pct_old_housing"] = (housing_clean["old_housing_units"] / housing_clean["total_units"] * 100).round(2)

print("Housing cleaned:")
print(housing_clean[["geoid", "total_units", "old_housing_units", "pct_old_housing"]].head())
print("\nMissing values:", housing_clean["pct_old_housing"].isna().sum())

Housing cleaned:
         geoid  total_units  old_housing_units  pct_old_housing
0  48113000100         1935               1087            56.18
1  48113000201         1314                966            73.52
2  48113000202         1943               1367            70.36
3  48113000300         2323               1057            45.50
4  48113000401         2271                535            23.56

Missing values: 3


In [16]:
# clean shapefile
dallas_clean = dallas_tracts.copy()

# drop unneeded columns
dallas_clean = dallas_clean.drop(columns=["STATEFP", "COUNTYFP", "TRACTCE", 
    "NAME", "NAMELSAD", "MTFCC", "FUNCSTAT"])

# rename columns
dallas_clean.columns = ["geoid", "aland", "awater", "intptlat", "intptlon", "geometry"]

# convert lat/lon to float
dallas_clean["intptlat"] = dallas_clean["intptlat"].astype(float)
dallas_clean["intptlon"] = dallas_clean["intptlon"].astype(float)

# calculate land area in square kilometers
dallas_clean["area_sqkm"] = (dallas_clean["aland"] / 1_000_000).round(4)

print("Shapefile cleaned:")
print(dallas_clean.info())
print("\n")
print(dallas_clean.head())

Shapefile cleaned:
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 645 entries, 176 to 6796
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   geoid      645 non-null    str     
 1   aland      645 non-null    int64   
 2   awater     645 non-null    int64   
 3   intptlat   645 non-null    float64 
 4   intptlon   645 non-null    float64 
 5   geometry   645 non-null    geometry
 6   area_sqkm  645 non-null    float64 
dtypes: float64(3), geometry(1), int64(2), str(1)
memory usage: 40.3 KB
None


           geoid    aland  awater   intptlat   intptlon  \
176  48113016638  3449205   17743  32.629589 -96.875812   
264  48113013630   292256       0  32.944479 -96.806708   
841  48113012211  1318895       0  32.802221 -96.689035   
842  48113012400  3053580       0  32.828229 -96.684809   
843  48113016513  5281302       0  32.635286 -96.933544   

                                              geometry  area_sqkm  


In [17]:
# clean landsat - mask no data pixels and convert LST to celsius
lst_clean = lst_data.copy()

# mask out no data pixels (value of 0)
lst_clean[lst_clean == 0] = np.nan

# apply USGS scale factor to convert to celsius
lst_clean = lst_clean * 0.00341802 + 149.0 - 273.15

print("LST converted to Celsius:")
print(f"Min temp: {np.nanmin(lst_clean):.2f} C")
print(f"Max temp: {np.nanmax(lst_clean):.2f} C")
print(f"Mean temp: {np.nanmean(lst_clean):.2f} C")
print(f"No data pixels: {np.isnan(lst_clean).sum()}")

# calculate NDVI with no data mask
b4_clean = b4_data.copy().astype(float)
b5_clean = b5_data.copy().astype(float)

# mask zero pixels
b4_clean[b4_clean == 0] = np.nan
b5_clean[b5_clean == 0] = np.nan

# calculate NDVI
ndvi_clean = (b5_clean - b4_clean) / (b5_clean + b4_clean)

print("\nNDVI calculated:")
print(f"Min NDVI: {np.nanmin(ndvi_clean):.3f}")
print(f"Max NDVI: {np.nanmax(ndvi_clean):.3f}")
print(f"Mean NDVI: {np.nanmean(ndvi_clean):.3f}")

LST converted to Celsius:
Min temp: -3.44 C
Max temp: 63.24 C
Mean temp: 37.90 C
No data pixels: 17472931

NDVI calculated:
Min NDVI: -0.218
Max NDVI: 0.777
Mean NDVI: 0.336


In [18]:
# clean enviroatlas
impervious_clean = impervious.copy()
canopy_clean = canopy.copy()

# drop objectid column
impervious_clean = impervious_clean.drop(columns=["OBJECTID"])
canopy_clean = canopy_clean.drop(columns=["OBJECTID"])

# rename columns
impervious_clean.columns = ["huc12", "pct_impervious"]
canopy_clean.columns = ["huc12", "pct_canopy"]

# drop missing values
impervious_clean = impervious_clean.dropna()

# merge impervious and canopy together
enviro_clean = impervious_clean.merge(canopy_clean, on="huc12")

print("EnviroAtlas cleaned:")
print(enviro_clean.info())
print("\n")
print(enviro_clean.describe())
print("\nMissing values:")
print(enviro_clean.isna().sum())

EnviroAtlas cleaned:
<class 'pandas.DataFrame'>
RangeIndex: 82871 entries, 0 to 82870
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   huc12           82871 non-null  int64  
 1   pct_impervious  82871 non-null  float64
 2   pct_canopy      82871 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 1.9 MB
None


              huc12  pct_impervious    pct_canopy
count  8.287100e+04    82871.000000  82871.000000
mean   1.002488e+11        1.963675     24.398222
std    4.995638e+10        5.384863     26.256122
min    1.010002e+10        0.000000      0.000000
25%    5.120204e+10        0.180947      0.646858
50%    1.018001e+11        0.541063     13.183416
75%    1.407000e+11        1.342630     45.969968
max    1.810020e+11       82.318146     95.899207

Missing values:
huc12             0
pct_impervious    0
pct_canopy        0
dtype: int64


In [19]:
import os

processed_path = r"C:\Users\egkbo\dallas_heat_island\data\processed"

# save cleaned census tables
income_clean.to_csv(os.path.join(processed_path, "income_clean.csv"), index=False)
population_clean.to_csv(os.path.join(processed_path, "population_clean.csv"), index=False)
housing_clean.to_csv(os.path.join(processed_path, "housing_clean.csv"), index=False)

# save cleaned shapefile
dallas_clean.to_file(os.path.join(processed_path, "dallas_tracts_clean.gpkg"), driver="GPKG")

# save cleaned enviroatlas
enviro_clean.to_csv(os.path.join(processed_path, "enviro_clean.csv"), index=False)

# save cleaned landsat arrays
import numpy as np
np.save(os.path.join(processed_path, "lst_celsius.npy"), lst_clean)
np.save(os.path.join(processed_path, "ndvi.npy"), ndvi_clean)

print("All cleaned datasets saved to data/processed/")

All cleaned datasets saved to data/processed/


In [20]:
# reload correct landsat files
lst_path = os.path.join(data_path, "LC08_L2SP_027037_20230726_20230805_02_T1_ST_B10.TIF")
b4_path = os.path.join(data_path, "LC08_L2SP_027037_20230726_20230805_02_T1_SR_B4.TIF")
b5_path = os.path.join(data_path, "LC08_L2SP_027037_20230726_20230805_02_T1_SR_B5.TIF")

with rasterio.open(lst_path) as src:
    lst_data = src.read(1).astype("float32")
    lst_crs = src.crs

with rasterio.open(b4_path) as src:
    b4_data = src.read(1).astype("float32")

with rasterio.open(b5_path) as src:
    b5_data = src.read(1).astype("float32")

# clean and convert
lst_clean = lst_data.copy()
lst_clean[lst_clean == 0] = np.nan
lst_clean = lst_clean * 0.00341802 + 149.0 - 273.15

b4_clean = b4_data.copy().astype(float)
b5_clean = b5_data.copy().astype(float)
b4_clean[b4_clean == 0] = np.nan
b5_clean[b5_clean == 0] = np.nan
ndvi_clean = (b5_clean - b4_clean) / (b5_clean + b4_clean)

print(f"LST mean: {np.nanmean(lst_clean):.2f} C")
print(f"NDVI mean: {np.nanmean(ndvi_clean):.3f}")

# save updated files
np.save(os.path.join(processed_path, "lst_celsius.npy"), lst_clean)
np.save(os.path.join(processed_path, "ndvi.npy"), ndvi_clean)
print("Saved updated Landsat files!")

LST mean: 42.25 C
NDVI mean: 0.247
Saved updated Landsat files!
